# 2 — Retrieval and Generation

Queries the index built by notebook 1. Runs per request: no parsing, no corpus
embedding.

```
query ─▶ transform ─▶ retrieve ─▶ rerank ─▶ diversify ─▶ compress ─▶ answer
```

| § | Stage |
|---|---|
| 0 | Setup and manifest check |
| 1 | Retrieval |
| 2 | Tables and figures |
| 3 | Query transformation |
| 4 | Context compression |
| 5 | Assembly and generation |
| 6 | Evaluation |
| 7 | Cost and latency |

---
## 0. Setup

`read_manifest()` compares this notebook's configuration against how the index was
built. Querying an index with a different embedding model produces well-formed
results and plausible scores that happen to be meaningless, so the check is an
assertion rather than a warning.

In [ ]:
%pip install -q "pinecone>=6.0" "openai>=1.60" tiktoken pandas numpy

In [ ]:
import os

os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("PINECONE_API_KEY", "")

In [ ]:
import json, re, time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from rag_common import (
    EMBED_MODEL, ENCODING, LLM_MODEL, NAMESPACE, RERANK_MODEL,
    client, embed, open_index, pc, read_manifest,
)

manifest = read_manifest()
index = open_index()

print(f"index      : {manifest['index_name']} ({manifest['embed_dims']}d {manifest['metric']})")
print(f"embedding  : {manifest['embed_model']}")
print(f"chunk size : {manifest['chunk_tokens']} tokens")
print(f"documents  : {list(manifest['documents'])}")

---
## 1. Retrieval

Three stages, each fixing a different failure:

1. **Dense retrieval** over a wide candidate set — recall matters here, not precision.
2. **Reranking** with a cross-encoder, which reads query and passage together instead
   of comparing two independent vectors. Precision is recovered here.
3. **Maximal marginal relevance** to drop near-duplicates, so adjacent chunks and
   repeated boilerplate do not consume the context budget with the same text twice.

The candidate pool is derived from the requested result count: reranking can only
reorder what retrieval returned.

In [ ]:
# Candidates per final result. Larger recovers more at higher rerank cost; below
# roughly 4x the reranker has too little to reorder.
CANDIDATE_MULTIPLIER = 6
MIN_CANDIDATES = 24

METADATA_FIELDS = ("chunk_id", "content_type", "page", "section_id", "doc_date", "text")
EXTRA_FIELDS = ("prev_id", "next_id", "table_id", "image_uri")


def _query_vector(text: str) -> np.ndarray:
    return embed([text])[0]


def _row(meta: dict, dense: float = 0.0, rerank: float = 0.0) -> dict:
    return {
        **{k: meta[k] for k in METADATA_FIELDS},
        **{k: meta.get(k) for k in EXTRA_FIELDS},
        "position": int(meta["position"]), "n_tokens": int(meta["n_tokens"]),
        "dense": dense, "rerank": rerank,
    }


def retrieve(query: str, top_k: int = 5, filters: dict | None = None,
             groups: list[str] | None = None, candidates: int | None = None) -> list[dict]:
    pool = candidates or max(MIN_CANDIDATES, top_k * CANDIDATE_MULTIPLIER)

    merged = dict(filters or {})
    if groups:
        merged["access"] = {"$in": groups}

    matches = index.query(
        vector=_query_vector(query).tolist(), top_k=pool, namespace=NAMESPACE,
        include_metadata=True, include_values=True, filter=merged or None,
    ).matches
    if not matches:
        return []

    indexed_model = matches[0].metadata.get("embed_model")
    if indexed_model and indexed_model != EMBED_MODEL:
        raise ValueError(f"index built with {indexed_model}, querying with {EMBED_MODEL}")

    ranked = pc.inference.rerank(
        model=RERANK_MODEL, query=query,
        documents=[m.metadata["text"] for m in matches],
        top_n=min(top_k * 2, len(matches)), return_documents=False,
    )

    results = []
    for entry in ranked.data:
        match = matches[entry.index]
        row = _row(match.metadata, round(match.score, 4), round(entry.score, 4))
        row["embedding"] = match.values
        results.append(row)
    return results

In [ ]:
def mmr(results: list[dict], top_k: int, diversity: float = 0.3) -> list[dict]:
    """Trade relevance against novelty so near-duplicates do not crowd out
    complementary passages. diversity=0 preserves rerank order."""
    if not results:
        return []
    vectors = np.asarray([r["embedding"] for r in results], dtype=np.float32)
    vectors /= np.clip(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-9, None)

    scores = np.asarray([r["rerank"] for r in results], dtype=np.float32)
    spread = float(scores.max() - scores.min())
    scores = (scores - scores.min()) / spread if spread > 1e-9 else np.ones_like(scores)

    selected = [int(scores.argmax())]
    while len(selected) < min(top_k, len(results)):
        similarity = (vectors @ vectors[selected].T).max(axis=1)
        objective = (1 - diversity) * scores - diversity * similarity
        objective[selected] = -np.inf
        selected.append(int(objective.argmax()))
    return [results[i] for i in selected]


def expand(results: list[dict], window: int = 1) -> list[dict]:
    """Follow the reading-order links written at ingestion.

    Content-addressed ids cannot be derived from position, so neighbours are reached
    through prev_id/next_id. Cost is proportional to the number of results, not to
    document length.
    """
    if not results or window <= 0:
        return results

    frontier = {r["chunk_id"]: r for r in results}
    collected = dict(frontier)
    for _ in range(window):
        wanted = {n for hit in frontier.values()
                  for n in (hit.get("prev_id"), hit.get("next_id"))
                  if n and n not in collected}
        if not wanted:
            break
        fetched = index.fetch(ids=sorted(wanted), namespace=NAMESPACE).vectors
        frontier = {}
        for vector in fetched.values():
            entry = _row(vector.metadata)
            entry["embedding"] = None
            collected[entry["chunk_id"]] = entry
            frontier[entry["chunk_id"]] = entry
    return sorted(collected.values(), key=lambda r: r["position"])


def search(query: str, top_k: int = 5, diversity: float = 0.3, **kwargs) -> list[dict]:
    return mmr(retrieve(query, top_k=top_k, **kwargs), top_k=top_k, diversity=diversity)

In [ ]:
QUESTION = "How fast is AI adoption growing across industries?"

for hit in search(QUESTION):
    print(f"[rerank {hit['rerank']:.3f} | dense {hit['dense']:.3f}] "
          f"p{hit['page']} {hit['content_type']}")
    print(f"  {hit['text'][:170].replace(chr(10), ' ')}\n")

The `dense` and `rerank` columns rarely agree. You will regularly see the reranker's
top result sitting at dense rank 8 or 15, meaning a plain `top_k=5` would have thrown
away the correct chunk.

---
## 2. Tables and figures

Table summaries are what natural-language queries match; the raw rows carry the exact
values. `with_table_rows()` walks from a matched summary to its fragments using the
shared `table_id`, so the model gets both the description and the numbers.

In [ ]:
def with_table_rows(results: list[dict], max_rows: int = 6) -> list[dict]:
    """Attach the raw fragments of any table whose summary was retrieved."""
    table_ids = {r["table_id"] for r in results
                 if r["content_type"] == "table_summary" and r.get("table_id")}
    if not table_ids:
        return results

    merged = {r["chunk_id"]: r for r in results}
    for table_id in table_ids:
        # A filtered query with a neutral vector: the filter selects the fragments,
        # the ranking among them does not matter because they are all wanted.
        fragments = index.query(
            vector=[0.0] * manifest["embed_dims"], top_k=max_rows,
            namespace=NAMESPACE, include_metadata=True,
            filter={"table_id": {"$eq": table_id},
                    "content_type": {"$eq": "table"}},
        ).matches
        for match in fragments:
            row = _row(match.metadata)
            row["embedding"] = None
            merged.setdefault(row["chunk_id"], row)
    return sorted(merged.values(), key=lambda r: r["position"])


hits = search("what does the schedule of assessments contain?", top_k=3)
for row in with_table_rows(hits):
    print(f"[{row['position']:>3}] {row['content_type']:<14} p{row['page']}  "
          f"{row['text'][:90].replace(chr(10), ' ')}")

In [ ]:
# Filtering on the typed metadata written at ingestion.
print("── table summaries only ──")
for h in search("enrolment by site", filters={"content_type": {"$eq": "table_summary"}}):
    print(f"  [{h['rerank']:.3f}] p{h['page']}  {h['text'][:90]}")

print("\n── figures, with their stored image ──")
for h in search("chart showing growth over time",
                filters={"content_type": {"$eq": "figure"}}):
    print(f"  [{h['rerank']:.3f}] p{h['page']}  {h.get('image_uri') or 'no image stored'}")
    print(f"      {h['text'][:110]}")

print("\n── recent documents only ──")
for h in search("adoption trends", filters={"doc_date": {"$gte": "2025"}}):
    print(f"  [{h['rerank']:.3f}] {h['doc_date']}  p{h['page']}")

---
## 3. Query transformation

A user's phrasing and a document's phrasing are different distributions.

**Multi-query** issues several rewordings and fuses the rankings with reciprocal rank
fusion, which combines lists without needing comparable scores across runs.

**HyDE** embeds a hypothetical answer instead of the question, on the grounds that
answers resemble passages and questions do not. It helps on short or underspecified
queries and hurts when the hypothetical drifts from the corpus, so it stays behind
the evaluation harness rather than on by default.

In [ ]:
# Reciprocal rank fusion constant, from the original formulation. Damps the influence
# of top ranks so agreement across queries outweighs any single list.
RRF_K = 60


def rewrite(query: str, n: int = 3) -> list[str]:
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[
            {"role": "system", "content":
             f"Rewrite the question {n} ways, varying vocabulary and specificity to "
             "match how a formal report might phrase it. One per line, no numbering."},
            {"role": "user", "content": query},
        ],
    )
    lines = [l.strip() for l in response.choices[0].message.content.splitlines() if l.strip()]
    return [query] + lines[:n]


def hyde(query: str) -> str:
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0, max_tokens=200,
        messages=[
            {"role": "system", "content":
             "Write a short factual passage as it would appear in a report answering "
             "the question. Invent plausible specifics; this is a retrieval probe and "
             "is never shown to the user."},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


def fused_search(query: str, top_k: int = 5, use_hyde: bool = False,
                 n_variants: int = 3, **kwargs) -> list[dict]:
    queries = rewrite(query, n_variants) if n_variants else [query]
    if use_hyde:
        queries.append(hyde(query))

    fused, seen = defaultdict(float), {}
    for variant in queries:
        for rank, hit in enumerate(retrieve(variant, top_k=top_k, **kwargs), start=1):
            fused[hit["chunk_id"]] += 1.0 / (RRF_K + rank)
            seen[hit["chunk_id"]] = hit
    order = sorted(fused, key=fused.get, reverse=True)
    return [seen[cid] for cid in order[:top_k]]

---
## 4. Context compression

Retrieved chunks are not the same as context. A 1024-token chunk retrieved for one
sentence carries most of a page of unrelated text, which costs money, adds latency,
and dilutes attention across the prompt.

**Extractive** ranks sentences within each chunk against the query embedding — no
extra LLM call, and sentence vectors reuse the embedding cache.

**Abstractive** asks a small model for the relevant spans. Better at implicit
relevance, at one call per chunk.

Table fragments are left alone by both: dropping rows from a table to save tokens
destroys the thing that made it worth retrieving.

In [ ]:
SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+|\n{2,}")
LITERAL_TYPES = {"table", "formula", "code"}


def compress_extractive(query: str, results: list[dict], keep: float = 0.5,
                        min_sentences: int = 2) -> list[dict]:
    query_vector = np.asarray(_query_vector(query), dtype=np.float32)
    query_vector /= np.linalg.norm(query_vector) + 1e-9

    # Sentences from every chunk are embedded in one request; per-chunk calls would
    # add one round trip per result and dominate query latency.
    split, flat = [], []
    for hit in results:
        if hit["content_type"] in LITERAL_TYPES:
            split.append(None)
            continue
        sentences = [s.strip() for s in SENTENCE_SPLIT.split(hit["text"]) if s.strip()]
        split.append(sentences if len(sentences) > min_sentences else None)
        if split[-1]:
            flat.extend(sentences)

    vectors = embed(flat) if flat else np.empty((0, query_vector.shape[0]), np.float32)
    if len(vectors):
        vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)

    compressed, cursor = [], 0
    for hit, sentences in zip(results, split):
        if sentences is None:
            compressed.append(hit)
            continue
        scores = vectors[cursor:cursor + len(sentences)] @ query_vector
        cursor += len(sentences)
        n_keep = max(min_sentences, int(len(sentences) * keep))
        text = " ".join(sentences[i] for i in sorted(np.argsort(scores)[-n_keep:]))
        compressed.append({**hit, "text": text, "n_tokens": len(ENCODING.encode(text))})
    return compressed


def compress_abstractive(query: str, results: list[dict]) -> list[dict]:
    compressed = []
    for hit in results:
        if hit["content_type"] in LITERAL_TYPES:
            compressed.append(hit)
            continue
        response = client.chat.completions.create(
            model=LLM_MODEL, temperature=0,
            messages=[
                {"role": "system", "content":
                 "Return only the sentences from the passage that bear on the "
                 "question, verbatim and in original order. Return NONE if nothing "
                 "is relevant."},
                {"role": "user", "content": f"Question: {query}\n\nPassage:\n{hit['text']}"},
            ],
        )
        text = response.choices[0].message.content.strip()
        if text.upper().startswith("NONE"):
            continue
        compressed.append({**hit, "text": text, "n_tokens": len(ENCODING.encode(text))})
    return compressed

---
## 5. Assembly and generation

Assembly fills a fixed token budget in relevance order, then restores document order
within the selection: prompt size stays bounded regardless of retrieval depth, and
the model reads passages in the order they were written.

In [ ]:
# Share of the generation model's window reserved for retrieved context. The rest
# covers the system prompt, the question, and the completion.
CONTEXT_BUDGET_TOKENS = 6000


def assemble(results: list[dict], budget: int = CONTEXT_BUDGET_TOKENS) -> tuple[str, dict]:
    chosen, used = [], 0
    for hit in results:
        if used + hit["n_tokens"] > budget:
            continue
        chosen.append(hit)
        used += hit["n_tokens"]
    chosen.sort(key=lambda h: h["position"])
    context = "\n\n".join(f"[{h['chunk_id']} p{h['page']}]\n{h['text']}" for h in chosen)
    return context, {"chunks": len(chosen), "tokens": used,
                     "dropped": len(results) - len(chosen)}


def answer(query: str, top_k: int = 5, compression: str | None = "extractive",
           expand_window: int = 0, table_rows: bool = True,
           use_fusion: bool = False, **kwargs) -> dict:
    started = time.time()
    results = (fused_search if use_fusion else search)(query, top_k=top_k, **kwargs)
    if not results:
        return {"answer": "No relevant context found.", "sources": [], "stats": {}}

    if table_rows:
        results = with_table_rows(results)
    if expand_window:
        results = expand(results, expand_window)
    if compression == "extractive":
        results = compress_extractive(query, results)
    elif compression == "abstractive":
        results = compress_abstractive(query, results)

    context, stats = assemble(results)
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[
            {"role": "system", "content":
             "Answer from the provided context only. Cite pages as [p12]. State "
             "plainly if the context does not contain the answer."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
    )
    stats["latency_s"] = round(time.time() - started, 2)
    return {"answer": response.choices[0].message.content,
            "sources": [(h["chunk_id"], h["page"], h["content_type"]) for h in results],
            "images": [h["image_uri"] for h in results if h.get("image_uri")],
            "stats": stats}

In [ ]:
result = answer(QUESTION)
print(result["stats"], "\n")
print(result["answer"])
if result["images"]:
    print("\nfigures cited:", *result["images"], sep="\n  ")

---
## 6. Evaluation

Every parameter above — chunk size, candidate multiplier, diversity, compression
ratio, table summarisation itself — is a hypothesis until it moves a number on a
labelled set.

Build 30 to 50 pairs by hand: read the document, write questions a real user would
ask, record which chunks answer them. Tag each with the content type you expect to
answer it. Include questions whose answers exist only in a table or only in a figure;
those are what discriminate between extraction configurations.

In [ ]:
def candidate_ids(query: str, k: int = 5) -> None:
    """Surface chunk ids while building the labelled set."""
    for hit in search(query, top_k=k):
        print(f"{hit['chunk_id']}  p{hit['page']}  {hit['content_type']}")
        print(f"   {hit['text'][:120].replace(chr(10), ' ')}")


candidate_ids("What are the eligibility criteria?")

In [ ]:
GOLD_PATH = Path("gold_set.json")

if GOLD_PATH.exists():
    GOLD = json.loads(GOLD_PATH.read_text())
else:
    GOLD = [
        {"query": "How fast is AI adoption growing?",   "gold": [], "type": "text"},
        {"query": "What are the eligibility criteria?", "gold": [], "type": "table_summary"},
        {"query": "What does the trend chart show?",    "gold": [], "type": "figure"},
    ]
    GOLD_PATH.write_text(json.dumps(GOLD, indent=2))

labelled = [row for row in GOLD if row.get("gold")]
print(f"{len(GOLD)} questions, {len(labelled)} labelled — target 30 to 50")

In [ ]:
def evaluate(gold: list[dict], k: int = 5, retriever=None, **kwargs) -> dict:
    """recall@k and MRR, overall and by expected content type."""
    retriever = retriever or search
    rows = [r for r in gold if r.get("gold")]
    if not rows:
        return {"error": "no labelled rows"}

    hits, reciprocal, by_type = [], [], defaultdict(list)
    for row in rows:
        ids = [h["chunk_id"] for h in retriever(row["query"], top_k=k, **kwargs)]
        target = set(row["gold"])
        found = int(any(i in target for i in ids))
        rank = next((n for n, i in enumerate(ids, 1) if i in target), None)
        hits.append(found)
        reciprocal.append(1 / rank if rank else 0.0)
        by_type[row.get("type", "text")].append(found)

    return {"n": len(rows),
            f"recall@{k}": round(sum(hits) / len(hits), 3),
            "mrr": round(sum(reciprocal) / len(reciprocal), 3),
            "by_type": {t: round(sum(v) / len(v), 3) for t, v in sorted(by_type.items())}}


def sweep(gold: list[dict]) -> pd.DataFrame:
    configurations = {
        "dense only":     dict(retriever=lambda q, top_k, **kw: retrieve(q, top_k=top_k)[:top_k]),
        "dense + rerank": dict(diversity=0.0),
        "+ mmr":          dict(diversity=0.3),
        "+ query fusion": dict(retriever=fused_search, diversity=0.3),
    }
    rows = []
    for name, kwargs in configurations.items():
        try:
            rows.append({"config": name, **evaluate(gold, **kwargs)})
        except Exception as exc:
            rows.append({"config": name, "error": str(exc)[:60]})
    return pd.DataFrame(rows)


sweep(GOLD)

### The experiment worth running first

Table summarisation is the one change in this pipeline that has not been measured.
Ingest the same documents twice — once as built, once with
`ingest.TABLE_SUMMARY_MIN_CHARS` set impossibly high so no summaries are produced —
and compare `by_type["table"]` and `by_type["table_summary"]` recall.

Then do the same for chunk size:

```bash
CHUNK_TOKEN_TARGET=512  python ingest.py --pdf x.pdf
CHUNK_TOKEN_TARGET=2048 python ingest.py --pdf x.pdf
```

Both are single numbers in a config. Neither has a defensible answer without your own
corpus behind it.

---
## 7. Cost and latency

Retrieval quality is one axis; cost per query and time to answer are the others. A
configuration that adds two points of recall for four times the latency is a
different trade than the recall number alone suggests.

In [ ]:
def profile(query: str, configurations: dict) -> pd.DataFrame:
    rows = []
    for name, kwargs in configurations.items():
        result = answer(query, **kwargs)
        rows.append({"config": name, **result["stats"]})
    return pd.DataFrame(rows)


profile(QUESTION, {
    "baseline":      dict(compression=None, table_rows=False),
    "+ table rows":  dict(compression=None),
    "+ extractive":  dict(compression="extractive"),
    "+ abstractive": dict(compression="abstractive"),
    "+ fusion":      dict(compression="extractive", use_fusion=True),
    "+ neighbours":  dict(compression="extractive", expand_window=1),
})

### Extending this

**Sparse retrieval.** Fit a BM25 encoder on the corpus and store sparse vectors
alongside dense ones at ingestion. Exact-match terms — trial identifiers, exhibit
numbers, defined terms — are where dense retrieval is weakest, and the index metric
already supports it.

**Parent retrieval.** Return whole sections at generation time rather than chunks.
`section_id` is written at ingestion for exactly this.

**Agentic retrieval.** Let the model decide whether to retrieve, write its own query,
judge the results, and search again. It calls `search()` as a tool, so this notebook
is the substrate rather than something it replaces.